In [ ]:
import numpy as np
import torch
from torch import Generator
from occhio import ToyModel

from occhio.autoencoder import TiedLinearRelu
from occhio.distributions.sparse import SparseUniform
from occhio.model_grid import Axis, ModelGrid
from occhio.visualization.geometry import plot_geometry, GeometryPlotComponent

In [ ]:
generator = torch.Generator("cpu").manual_seed(42)

In [ ]:
N_FEATURES = 400
N_HIDDEN = 30
FEATURE_IMPORTANCE_DECAY = 0.996

In [ ]:
def create_model(params):
    device = "mps"
    generator = Generator(device=device).manual_seed(42)

    return ToyModel(
        distribution=SparseUniform(
            n_features=N_FEATURES,
            p_active=params["Feature Probability"],
            generator=generator,
            device=device,
        ),
        ae=TiedLinearRelu(
            n_features=N_FEATURES, n_hidden=N_HIDDEN, generator=generator, device=device
        ),
        importances=torch.tensor(
            [FEATURE_IMPORTANCE_DECAY**i for i in range(N_FEATURES)]
        ),
        device=device,
    )


model_grid = ModelGrid(
    create_model,
    axes=[Axis(label="Feature Probability", values=10 ** np.linspace(0, -3, 8))],
)

In [ ]:
model_grid.fit(n_epochs=15000)

In [ ]:
import importlib
import occhio.visualization.geometry

importlib.reload(occhio.visualization.geometry)
from occhio.visualization.geometry import plot_geometry

fig = plot_geometry(model_grid)

In [ ]:
fig.show()